In [1]:
import os
import re
import gc
import math
import random
import string
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, f1_score
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
try:
    from transformers import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

# =========================================================
# 0. CONFIG
# =========================================================
SEED = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.15

EPOCHS_PROPOSED = 6
EPOCHS_BASELINE = 6
MAX_LEN = 256

LR_BASE = 2e-5
LR_HEAD = 5e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
PATIENCE = 2
LABEL_SMOOTHING_PROPOSED = 0.05
LABEL_SMOOTHING_BASELINE = 0.02

# Dataset quality rules, aligned with notebook (1).
MIN_HUMAN_AUTHORS = 8
MAX_HUMAN_AUTHORS = 12
MIN_HUMAN_RAW_ROWS = 800
MAX_HUMAN_RAW_ROWS = 2000
MIN_HUMAN_ROWS_PER_AUTHOR = 12
MAX_AI_RATIO_TO_HUMAN = 0.25
CHUNKS_PER_HUMAN_TEXT = 2

SOURCE_FILENAMES = [
    "detik.csv",
    "kumparan.csv",
    "tempo.csv",
    "mojok.csv",
    "tirto.csv",
    "synthetic_imposter_dataset.csv",
]

CANDIDATE_BASE_PATHS = [
    "/kaggle/input/datasets/xandertrevor/stylometry/raw/",
    "/kaggle/input/stylometry/raw/",
    "/kaggle/input/",
    "/mnt/data/",
    ".",
]

WINDOWS_FALLBACKS = {
    "detik.csv": r"D:\StyloGuard\backend\data\raw\detik.csv",
    "kumparan.csv": r"D:\StyloGuard\backend\data\raw\kumparan.csv",
    "mojok.csv": r"D:\StyloGuard\backend\data\raw\mojok.csv",
    "tempo.csv": r"D:\StyloGuard\backend\data\raw\tempo.csv",
    "tirto.csv": r"D:\StyloGuard\backend\data\raw\tirto.csv",
    "synthetic_imposter_dataset.csv": r"D:\StyloGuard (Branch)\StyloGuard\backend\data\raw\synthetic_imposter_dataset.csv",
}

PROPOSED_MODEL_NAME = "indobenchmark/indobert-base-p1"
BASELINE_MODELS = [
    ("indobenchmark/indobert-base-p1", "IndoBERT (CLS only)"),
    ("microsoft/mdeberta-v3-base", "mDeBERTa-v3"),
    ("xlm-roberta-base", "XLM-RoBERTa"),
    ("bert-base-multilingual-cased", "mBERT"),
]

SHOW_TQDM = True

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)
torch.backends.cudnn.benchmark = True
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()
BATCH_SIZE = 16 if N_GPU >= 2 else 8
NUM_WORKERS = 2 if os.name != "nt" else 0
USE_AMP = False
USE_DATA_PARALLEL = N_GPU >= 2

# =========================================================
# 1. LOAD DATA
# =========================================================
def find_file(filename):
    for base in CANDIDATE_BASE_PATHS:
        path = Path(base) / filename
        if path.exists():
            return str(path)

    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        matches = sorted(kaggle_root.rglob(filename), key=lambda p: len(str(p)))
        if matches:
            return str(matches[0])

    fallback = Path(WINDOWS_FALLBACKS.get(filename, ""))
    if fallback.exists():
        return str(fallback)

    return None

def read_csv_safely(path):
    try:
        return pd.read_csv(path, encoding="utf-8-sig", encoding_errors="replace")
    except TypeError:
        return pd.read_csv(path, encoding="utf-8-sig")
    except Exception:
        try:
            return pd.read_csv(
                path,
                encoding="utf-8-sig",
                encoding_errors="replace",
                engine="python",
                on_bad_lines="skip",
            )
        except TypeError:
            return pd.read_csv(path, encoding="utf-8-sig", engine="python", on_bad_lines="skip")

def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

frames = []
for filename in SOURCE_FILENAMES:
    path = find_file(filename)
    if path is None:
        continue

    df = read_csv_safely(path)
    df.columns = [str(c).strip().lower() for c in df.columns]
    if "text" not in df.columns or "author" not in df.columns:
        continue

    df = df.copy()
    df["text"] = df["text"].astype(str).map(clean_text)
    df["author"] = df["author"].astype(str).map(lambda x: re.sub(r"\s+", " ", x).strip())
    df = df.dropna(subset=["text", "author"])
    df = df[df["text"].str.strip().ne("")]
    df = df[df["author"].str.lower().ne("author")]
    df = df[df["text"].str.split().str.len() >= 50]

    if filename == "synthetic_imposter_dataset.csv":
        df["author"] = "AI"

    frames.append(df[["author", "text"]].copy())

if not frames:
    raise RuntimeError("No valid datasets found. Attach the CSVs in Kaggle or update CANDIDATE_BASE_PATHS.")

df_master = pd.concat(frames, ignore_index=True)
df_master = df_master.drop_duplicates(subset=["author", "text"]).reset_index(drop=True)

# =========================================================
# 2. DATASET QUALITY FILTERING
# =========================================================
df_master["is_ai"] = df_master["author"].eq("AI")
human_df = df_master[~df_master["is_ai"]].copy()
ai_df = df_master[df_master["is_ai"]].copy()

human_counts = human_df["author"].value_counts()
eligible_counts = human_counts[human_counts >= MIN_HUMAN_ROWS_PER_AUTHOR]

selected_authors = []
running_rows = 0
for author, count in eligible_counts.items():
    if running_rows >= MAX_HUMAN_RAW_ROWS:
        break
    selected_authors.append(author)
    running_rows += int(count)
    if len(selected_authors) >= MIN_HUMAN_AUTHORS and running_rows >= MIN_HUMAN_RAW_ROWS:
        break
    if len(selected_authors) >= MAX_HUMAN_AUTHORS:
        break

if running_rows < MIN_HUMAN_RAW_ROWS:
    for author, count in eligible_counts.items():
        if author in selected_authors:
            continue
        selected_authors.append(author)
        running_rows += int(count)
        if running_rows >= MIN_HUMAN_RAW_ROWS or len(selected_authors) >= MAX_HUMAN_AUTHORS:
            break

kept_human_df = human_df[human_df["author"].isin(selected_authors)].copy()
if len(kept_human_df) < MIN_HUMAN_RAW_ROWS:
    raise RuntimeError(f"Only {len(kept_human_df)} human raw rows kept; need at least {MIN_HUMAN_RAW_ROWS}.")

if len(kept_human_df) > MAX_HUMAN_RAW_ROWS:
    trimmed_parts = []
    remaining = MAX_HUMAN_RAW_ROWS
    per_author_counts = kept_human_df["author"].value_counts()
    for author in selected_authors:
        author_rows = kept_human_df[kept_human_df["author"].eq(author)]
        n_take = min(len(author_rows), max(MIN_HUMAN_ROWS_PER_AUTHOR, remaining // max(1, len(selected_authors) - len(trimmed_parts))))
        n_take = min(n_take, remaining)
        trimmed_parts.append(author_rows.sample(n=n_take, random_state=SEED))
        remaining -= n_take
        if remaining <= 0:
            break
    kept_human_df = pd.concat(trimmed_parts, ignore_index=True)
    if len(kept_human_df) < MIN_HUMAN_RAW_ROWS:
        kept_human_df = human_df[human_df["author"].isin(selected_authors)].sample(
            n=MIN_HUMAN_RAW_ROWS,
            random_state=SEED,
        )

max_ai_rows = int(len(kept_human_df) * MAX_AI_RATIO_TO_HUMAN)
if len(ai_df) > 0 and max_ai_rows > 0:
    kept_ai_df = ai_df.sample(n=min(len(ai_df), max_ai_rows), random_state=SEED).copy()
else:
    kept_ai_df = ai_df.iloc[0:0].copy()

if len(kept_ai_df) > int(len(kept_human_df) * MAX_AI_RATIO_TO_HUMAN):
    raise RuntimeError("AI rows exceed the configured 25% human-row cap.")

final_df = pd.concat([kept_human_df, kept_ai_df], ignore_index=True)
final_df = final_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

labels = sorted(final_df["author"].unique().tolist())
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}
final_df["label"] = final_df["author"].map(label2id).astype(int)
NUM_CLASSES = len(labels)

if NUM_CLASSES < 2:
    raise RuntimeError("Need at least two classes after filtering.")

# =========================================================
# 3. STYLOMETRIC FEATURES
# =========================================================
WORD_RE = re.compile(r"[\w]+", re.UNICODE)
SENTENCE_RE = re.compile(r"[^.!?]+[.!?]*", re.UNICODE)
FUNCTION_WORDS = [
    "yang", "dan", "di", "ke", "dari", "dengan", "untuk", "pada",
    "ini", "itu", "tidak", "akan", "juga", "karena", "sebagai",
    "dalam", "adalah", "atau", "oleh", "agar", "bagi", "para",
    "saat", "setelah", "sebelum", "namun", "tetapi", "hingga",
]

try:
    import nltk
    from nltk.corpus import stopwords
    nltk.download("stopwords", quiet=True)
    INDONESIAN_STOPWORDS = set(stopwords.words("indonesian"))
except Exception:
    INDONESIAN_STOPWORDS = set()

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def tokenize_words(text):
    return [t.lower() for t in WORD_RE.findall(normalize_text(text))]

def split_sentences(text):
    sentences = [s.strip() for s in SENTENCE_RE.findall(normalize_text(text))]
    return [s for s in sentences if s]

def safe_div(n, d):
    return n / d if d else 0.0

def extract_stylometry(text):
    raw = str(text)
    words = tokenize_words(raw)
    sentences = split_sentences(raw)
    sentence_lengths = [len(tokenize_words(s)) for s in sentences]
    paragraphs = [p.strip() for p in raw.splitlines() if p.strip()]
    chars = [c for c in raw if not c.isspace()]
    punctuation = [c for c in raw if c in string.punctuation]
    uppercase_chars = [c for c in raw if c.isupper()]
    numeric_tokens = [w for w in words if w.isdigit()]
    stop_count = [w for w in words if w in INDONESIAN_STOPWORDS]

    wc = len(words)
    sc = len(sentences)
    cc = len(chars)
    avg_sent_len = safe_div(wc, sc)
    sent_var = safe_div(sum((x - avg_sent_len) ** 2 for x in sentence_lengths), len(sentence_lengths))

    feats = [
        float(wc),
        float(sc),
        safe_div(sum(len(w) for w in words), wc),
        avg_sent_len,
        sent_var,
        safe_div(len(set(words)), wc),
        safe_div(len(punctuation), cc),
        safe_div(raw.count(","), cc),
        safe_div(raw.count("."), cc),
        safe_div(raw.count("?"), cc),
        safe_div(raw.count("!"), cc),
        safe_div(raw.count(";") + raw.count(":"), cc),
        safe_div(raw.count("-"), cc),
        safe_div(sum(ch.isdigit() for ch in raw), cc),
        safe_div(len(uppercase_chars), cc),
        safe_div(len(numeric_tokens), wc),
        safe_div(len(stop_count), wc),
        float(len(paragraphs) or 1),
        safe_div(sum(len(tokenize_words(p)) for p in paragraphs), len(paragraphs) or 1),
        safe_div(sum(1 for w in words if len(w) <= 3), wc),
        safe_div(sum(1 for w in words if len(w) >= 8), wc),
        safe_div(sum(1 for w in words if w.endswith("nya")), wc),
        safe_div(sum(1 for w in words if w.endswith("lah")), wc),
        safe_div(sum(1 for w in words if w.endswith("kah")), wc),
    ]

    for fw in FUNCTION_WORDS:
        feats.append(safe_div(words.count(fw), wc))

    return feats

final_df["stylometry"] = final_df["text"].apply(extract_stylometry)
sty_matrix = np.array(final_df["stylometry"].tolist(), dtype=np.float32)

# =========================================================
# 4. SPLIT AT ARTICLE LEVEL
# =========================================================
X_text = final_df["text"].values
X_sty = sty_matrix
y = final_df["label"].values
author_names = final_df["author"].values
article_ids = np.arange(len(final_df))

X_train_text, X_test_text, X_train_sty, X_test_sty, y_train, y_test, author_train, author_test, id_train, id_test = train_test_split(
    X_text, X_sty, y, author_names, article_ids,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)

X_train_text, X_val_text, X_train_sty, X_val_sty, y_train, y_val, author_train, author_val, id_train, id_val = train_test_split(
    X_train_text, X_train_sty, y_train, author_train, id_train,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=y_train,
)

scaler = StandardScaler()
X_train_sty = scaler.fit_transform(X_train_sty).astype(np.float32)
X_val_sty = scaler.transform(X_val_sty).astype(np.float32)
X_test_sty = scaler.transform(X_test_sty).astype(np.float32)

# =========================================================
# 5. CHUNKING / DATA EXPANSION
# =========================================================
def chunk_text(text, n_chunks=2):
    words = str(text).split()
    if len(words) <= 1 or n_chunks <= 1:
        return [str(text).strip()]

    boundaries = np.linspace(0, len(words), n_chunks + 1, dtype=int)
    pieces = []
    for i in range(n_chunks):
        piece = " ".join(words[boundaries[i]:boundaries[i + 1]]).strip()
        if piece:
            pieces.append(piece)
    return pieces if pieces else [str(text).strip()]

def expand_split(texts, stylometry, labels, authors, ids, chunk_human=True):
    out_texts, out_sty, out_labels, out_authors, out_ids = [], [], [], [], []
    for text, sty, label, author, aid in zip(texts, stylometry, labels, authors, ids):
        should_chunk = chunk_human and (author != "AI")
        pieces = chunk_text(text, CHUNKS_PER_HUMAN_TEXT) if should_chunk else [str(text).strip()]
        for piece in pieces:
            out_texts.append(piece)
            out_sty.append(sty)
            out_labels.append(int(label))
            out_authors.append(author)
            out_ids.append(int(aid))
    return (
        out_texts,
        np.asarray(out_sty, dtype=np.float32),
        np.asarray(out_labels, dtype=np.int64),
        np.asarray(out_authors),
        np.asarray(out_ids, dtype=np.int64),
    )

X_train_text_c, X_train_sty_c, y_train_c, auth_train_c, id_train_c = expand_split(
    X_train_text, X_train_sty, y_train, author_train, id_train, chunk_human=True
)
X_val_text_c, X_val_sty_c, y_val_c, auth_val_c, id_val_c = expand_split(
    X_val_text, X_val_sty, y_val, author_val, id_val, chunk_human=True
)
X_test_text_c, X_test_sty_c, y_test_c, auth_test_c, id_test_c = expand_split(
    X_test_text, X_test_sty, y_test, author_test, id_test, chunk_human=True
)

# =========================================================
# 6. DATASET
# =========================================================
class AuthorshipDataset(Dataset):
    def __init__(self, texts, stylometry, labels, article_ids, tokenizer, max_len=MAX_LEN):
        self.texts = texts
        self.stylometry = stylometry
        self.labels = labels
        self.article_ids = article_ids
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "stylometry": torch.tensor(self.stylometry[idx], dtype=torch.float),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
            "article_id": torch.tensor(self.article_ids[idx], dtype=torch.long),
        }

def make_weighted_sampler(labels):
    counts = np.bincount(labels, minlength=NUM_CLASSES)
    counts = np.maximum(counts, 1)
    weights = np.array([1.0 / counts[label] for label in labels], dtype=np.float64)
    return WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=len(weights), replacement=True)

def make_loaders(tokenizer):
    train_ds = AuthorshipDataset(X_train_text_c, X_train_sty_c, y_train_c, id_train_c, tokenizer)
    val_ds = AuthorshipDataset(X_val_text_c, X_val_sty_c, y_val_c, id_val_c, tokenizer)
    test_ds = AuthorshipDataset(X_test_text_c, X_test_sty_c, y_test_c, id_test_c, tokenizer)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=make_weighted_sampler(y_train_c),
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    return train_ds, val_ds, test_ds, train_loader, val_loader, test_loader

# =========================================================
# 7. MODELS
# =========================================================
def masked_mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

class DualChannelIndoBERT(nn.Module):
    """
    Proposed model (Optimized Architecture):
    - IndoBERT CLS token (768)
    - Stylometric FFNN branch (64)
    - Concatenation -> Classifier (832 -> 256 -> num_classes)
    """
    def __init__(self, model_name, num_classes, num_sty_features):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size

        self.sty_branch = nn.Sequential(
            nn.Linear(num_sty_features, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(0.20),
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden + 64, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(256, num_classes),
        )

    def forward(self, input_ids, attention_mask, stylometry, return_attentions=False):
        if return_attentions:
            self.backbone.config.output_attentions = True
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, output_attentions=return_attentions)
        cls_vec = outputs.last_hidden_state[:, 0, :]
        sty_vec = self.sty_branch(stylometry)
        fused = torch.cat([cls_vec, sty_vec], dim=1)
        logits = self.classifier(fused)
        
        if return_attentions:
            return logits, outputs.attentions
        return logits

class CLSOnlyTransformer(nn.Module):
    """Unmodified baseline: a plain transformer encoder with CLS token and linear head."""
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(0.20)
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(hidden // 2, num_classes),
        )

    def forward(self, input_ids, attention_mask, stylometry=None, return_attentions=False):
        if return_attentions:
            self.backbone.config.output_attentions = True
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, output_attentions=return_attentions)
        cls_vec = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_vec)
        logits = self.classifier(x)
        if return_attentions:
            return logits, outputs.attentions
        return logits

def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def maybe_parallel(model):
    return nn.DataParallel(model) if USE_DATA_PARALLEL else model

# =========================================================
# 8. EVALUATION HELPERS
# =========================================================
@torch.no_grad()
def predict_grouped(model, loader, use_stylometry, ai_label_idx=None):
    model.eval()
    logits_by_group = defaultdict(list)
    label_by_group = {}

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].cpu().numpy()
        group_ids = batch["article_id"].cpu().numpy()

        if use_stylometry:
            stylometry = batch["stylometry"].to(device)
            outputs = model(input_ids, attention_mask, stylometry)
        else:
            outputs = model(input_ids, attention_mask)

        outputs = outputs.detach().cpu().numpy()

        for i, gid in enumerate(group_ids):
            logits_by_group[int(gid)].append(outputs[i])
            label_by_group[int(gid)] = int(labels[i])

    y_true = []
    y_pred = []
    for gid in sorted(logits_by_group.keys()):
        mean_logits = np.mean(logits_by_group[gid], axis=0)
        pred = int(np.argmax(mean_logits))
        y_pred.append(pred)
        y_true.append(label_by_group[gid])

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    
    ai_f1 = 0.0
    if ai_label_idx is not None:
        if ai_label_idx in y_true or ai_label_idx in y_pred:
            ai_f1 = f1_score(y_true, y_pred, labels=[ai_label_idx], average=None, zero_division=0)[0]
    
    return acc, prec, f1, ai_f1

def make_optimizer(model, proposed=False):
    base_model = unwrap_model(model)
    if proposed:
        backbone_params = list(base_model.backbone.parameters())
        head_params = (
            list(base_model.sty_branch.parameters()) +
            list(base_model.classifier.parameters())
        )
        return torch.optim.AdamW(
            [
                {"params": backbone_params, "lr": LR_BASE},
                {"params": head_params, "lr": LR_HEAD},
            ],
            weight_decay=WEIGHT_DECAY,
        )
    return torch.optim.AdamW(model.parameters(), lr=LR_BASE, weight_decay=WEIGHT_DECAY)

def train_one_model(model, train_loader, val_loader, use_stylometry, proposed=False, epochs=3):
    # Retrieve AI label index if exists
    ai_label_idx = label2id.get("AI", None)

    class_counts = np.bincount(y_train_c, minlength=NUM_CLASSES)
    class_counts = np.maximum(class_counts, 1)
    class_weights = torch.tensor(class_counts.sum() / (NUM_CLASSES * class_counts), dtype=torch.float, device=device)

    criterion = nn.CrossEntropyLoss(
        label_smoothing=LABEL_SMOOTHING_PROPOSED if proposed else LABEL_SMOOTHING_BASELINE,
    )

    optimizer = make_optimizer(model, proposed=proposed)
    total_steps = max(len(train_loader) * epochs, 1)
    warmup_steps = max(int(0.10 * total_steps), 1)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    best_val_f1 = -1.0
    best_state = None
    patience = 0

    for epoch in range(epochs):
        model.train()
        iterator = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}", leave=False, disable=not SHOW_TQDM, mininterval=5.0)

        train_loss = 0.0
        for batch in iterator:
            optimizer.zero_grad(set_to_none=True)

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_batch = batch["label"].to(device)

            if use_stylometry:
                stylometry = batch["stylometry"].to(device)
                outputs = model(input_ids, attention_mask, stylometry)
            else:
                outputs = model(input_ids, attention_mask)
            
            loss = criterion(outputs, labels_batch)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        
        # Calculate Train F1 and Val F1 to check for Overfitting
        _, _, train_f1, train_ai_f1 = predict_grouped(model, train_loader, use_stylometry, ai_label_idx)
        _, _, val_f1, val_ai_f1 = predict_grouped(model, val_loader, use_stylometry, ai_label_idx)
        
        print(f"Epoch {epoch + 1}/{epochs} | Train Loss: {train_loss:.4f} | Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | Val AI F1: {val_ai_f1:.4f}")
        
        # Overfitting Indicator
        if train_f1 - val_f1 > 0.15:
            print(f"  --> Warning: Possible overfitting detected (Train F1 is {train_f1 - val_f1:.4f} higher than Val F1)")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
            if patience >= PATIENCE:
                print(f"Early stopping at epoch {epoch + 1}!")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

def run_model(model_name, display_name, proposed=False):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_ds, val_ds, test_ds, train_loader, val_loader, test_loader = make_loaders(tokenizer)

    if proposed:
        model = DualChannelIndoBERT(model_name, NUM_CLASSES, X_train_sty_c.shape[1])
        epochs = EPOCHS_PROPOSED
        use_stylometry = True
    else:
        model = CLSOnlyTransformer(model_name, NUM_CLASSES)
        epochs = EPOCHS_BASELINE
        use_stylometry = False
    
    # Cast to float32 to prevent Half/Float issues, then move to device
    model = model.float().to(device)
    model = maybe_parallel(model)
    
    model = train_one_model(
        model,
        train_loader,
        val_loader,
        use_stylometry=use_stylometry,
        proposed=proposed,
        epochs=epochs,
    )

    ai_label_idx = label2id.get("AI", None)
    acc, prec, f1, ai_f1 = predict_grouped(model, test_loader, use_stylometry=use_stylometry, ai_label_idx=ai_label_idx)

    # We will return the model and tokenizer for the proposed model so it can be used for xAI and export
    result_dict = {
        "model_name": display_name,
        "accuracy": acc,
        "precision": prec,
        "f1_score": f1,
        "ai_f1_score": ai_f1
    }
    
    result_dict["trained_model"] = model
    result_dict["tokenizer"] = tokenizer
        
    del train_ds, val_ds, test_ds, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result_dict

# =========================================================
# 9. TRAIN BASELINE: XLM-RoBERTa
# =========================================================
results = []
print("\n--- Training Baseline: XLM-RoBERTa ---")
proposed_res = run_model("xlm-roberta-base", "XLM-RoBERTa", proposed=False)
proposed_model_obj = proposed_res.pop("trained_model")
proposed_tokenizer = proposed_res.pop("tokenizer")
results.append(proposed_res)

results_df = pd.DataFrame(results)
print("\nFINAL RESULTS ON TEST SET:")
print(results_df.to_string(index=False, formatters={
    "accuracy": "{:.4f}".format,
    "precision": "{:.4f}".format,
    "f1_score": "{:.4f}".format,
    "ai_f1_score": "{:.4f}".format,
}))

# =========================================================
# 10. EXPLAINABLE AI (xAI) - ATTENTION VISUALIZATION
# =========================================================
import matplotlib.pyplot as plt
import seaborn as sns

def explain_prediction(model, tokenizer, text, sty_features, true_label_name, label_map):
    # Unwrap model in case it is wrapped in nn.DataParallel
    base_model = unwrap_model(model)
    base_model.eval()

    # Swap backbone to eager mode for attention extraction
    # (PyTorch SDPA blocks output_attentions; eager computes them)
    model_id = base_model.backbone.config._name_or_path
    eager_backbone = AutoModel.from_pretrained(model_id, attn_implementation="eager")
    eager_backbone.load_state_dict(base_model.backbone.state_dict())
    eager_backbone.to(device)
    original_backbone = base_model.backbone
    base_model.backbone = eager_backbone

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
    sty_tensor = torch.tensor([sty_features], dtype=torch.float).to(device)

    with torch.no_grad():
        logits, attentions = base_model(input_ids, attention_mask, sty_tensor, return_attentions=True)
        pred_id = torch.argmax(logits, dim=1).item()

    # Restore original backbone
    base_model.backbone = original_backbone

    id2label_map = {v: k for k, v in label_map.items()}
    pred_label_name = id2label_map[pred_id]

    print(f"True Author: {true_label_name} | Predicted Author: {pred_label_name}")
    print("-" * 50)

    # Extract Attention from the last layer, averaged across heads
    last_layer_attention = attentions[-1][0]
    avg_attention = torch.mean(last_layer_attention, dim=0)

    # Attention of the [CLS] token towards all other tokens
    cls_attention = avg_attention[0].cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    token_att_pairs = [(tok, att) for tok, att in zip(tokens, cls_attention) if tok not in ["[CLS]", "[SEP]", "[PAD]"]]
    token_att_pairs.sort(key=lambda x: x[1], reverse=True)

    print("Top 10 Contextual Tokens Driving the Model's Decision:")
    for tok, att in token_att_pairs[:10]:
        print(f"{tok:15s} : {att:.4f}")

# =========================================================
# RUN xAI
# =========================================================
print("\nRunning xAI on a random sample from the test set...")
try:
    ai_indices = [i for i, auth in enumerate(auth_test_c) if auth == "AI"]
    sample_idx = random.choice(ai_indices) if ai_indices else random.randint(0, len(X_test_text_c) - 1)
except Exception:
    sample_idx = random.randint(0, len(X_test_text_c) - 1)

sample_text = X_test_text_c[sample_idx]
sample_sty = X_test_sty_c[sample_idx]
sample_author = auth_test_c[sample_idx]

explain_prediction(
    model=proposed_model_obj,
    tokenizer=proposed_tokenizer,
    text=sample_text,
    sty_features=sample_sty,
    true_label_name=sample_author,
    label_map=label2id
)


--- Training Baseline: XLM-RoBERTa ---


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Epoch 5/6 | Train Loss: 0.8374 | Train F1: 0.7922 | Val F1: 0.7213 | Val AI F1: 0.6829


Epoch 6/6:   0%|          | 0/82 [00:00<?, ?it/s]

Epoch 6/6 | Train Loss: 0.7355 | Train F1: 0.8521 | Val F1: 0.7996 | Val AI F1: 0.7907

FINAL RESULTS ON TEST SET:
 model_name accuracy precision f1_score ai_f1_score
XLM-RoBERTa   0.7606    0.7832   0.7484      0.8101

Running xAI on a random sample from the test set...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

True Author: AI | Predicted Author: Habib Allbi Ferdian
--------------------------------------------------
Top 10 Contextual Tokens Driving the Model's Decision:
▁Samsung        : 0.0064
HI              : 0.0059
HI              : 0.0057
HI              : 0.0057
<s>             : 0.0056
HI              : 0.0056
HI              : 0.0054
▁Samsung        : 0.0054
▁Samsung        : 0.0054
HI              : 0.0054
